# Twist Mapper GPU — Class-Based Pipeline Demo

This notebook runs the same tasks as `vectorized_code_trial.ipynb` (tile extraction from a zero-copy `as_strided` view, per-tile FFT, Bragg-ring-radius detection, angular intensity profile, twist-angle estimation via correlogram) but through the class-based rebuild in `twist_mapper_GPU/`:

- **`TiledVideo`** (`twist_mapper_GPU/tiled_video.py`) — owns the video tensor + zero-copy tile grid; per-stage plots for the raw tile, its placement on the frame, and its FFT.
- **`TileFFTAnalysis`** (`twist_mapper_GPU/tile_fft_analysis.py`) — Bragg-ring-radius → angular-profile → twist-angle pipeline for one tile, computing the tile's FFT once and caching every stage's result. Each stage is a `compute_*`/`plot_*` pair.

**Data:** runs on the sample image `data/moire.npz` by default — fetch it first with `python download_sample_data.py` (from this `examples/` folder; see `data/README.md` for the source and citation). To analyze your own data, set `SOURCE_PATH` below to any video, image, or `.npz` file.

In [ ]:
import torch

from twist_mapper_GPU import (TiledVideo, TileFFTAnalysis, load_video_tensor,
                              extract_pixel_size_ang, default_d_spacing_ang, get_material)

print(f"CUDA available: {torch.cuda.is_available()}")

## Load video + build the zero-copy tile grid

`load_video_tensor` loads the source (video, image, or `.npz`) straight to a grayscale tensor on the GPU (CPU if no GPU), and `TiledVideo` builds the `as_strided` 5D view once (`TiledVideo.from_path` does both in one call for video files) — `tile_size`/`stride` are set here instead of being re-passed to every call below.

In [ ]:
import os
SOURCE_PATH = os.path.join('data', 'moire.npz')  # or your own video / image / .npz
TILE_SIZE = 128
STRIDE = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# crop_black_margins=True strips black padding (e.g. a video's pillarbox
# bars) before tiling, so tiles never land on dead padding and plots don't
# show it. A still image or .npz loads as a 1-frame (or N-frame) video.
tiled = TiledVideo(load_video_tensor(SOURCE_PATH, device), tile_size=TILE_SIZE, stride=STRIDE,
                   crop_black_margins=True)
print(tiled)

## Stage 1–3: tile extraction, placement, and FFT (`TiledVideo`)

In [ ]:
FRAME_IDX = tiled.video.shape[0] // 2  # middle frame (0 for a single image)
ROW_IDX, COL_IDX = 2, 4

# Stage 1: the raw tile, pulled zero-copy from the 5D view
tiled.plot_tile(FRAME_IDX, ROW_IDX, COL_IDX)

In [ ]:
# Stage 2: where that tile sits within the full frame
tiled.plot_tile_on_frame(FRAME_IDX, ROW_IDX, COL_IDX)

In [ ]:
# Stage 3: the tile's 2D FFT magnitude, without and with Hann windowing
SUBTRACT_MEAN = False  # subtract each tile's mean before windowing/FFT — reduces
                       # DC, does not sharpen Bragg peaks (see
                       # TileFFTAnalysis._compute_fft_magnitude's docstring)
tiled.plot_tile_fft(FRAME_IDX, ROW_IDX, COL_IDX, hann=False, subtract_mean=SUBTRACT_MEAN)
tiled.plot_tile_fft(FRAME_IDX, ROW_IDX, COL_IDX, hann=True, subtract_mean=SUBTRACT_MEAN)

## Stage 4–6: Bragg ring radius → angular profile → twist angle (`TileFFTAnalysis`)

`tiled.analyze(...)` hands one tile off to a `TileFFTAnalysis`, which computes the tile's FFT magnitude **once** (cached on `.fft_magnitude`) instead of recomputing it for every stage the way the original notebook's free functions did.

In [ ]:
# Pixel size from the file's own metadata when it has any (.npz), else a manual fallback
PIXEL_SIZE_ANG = extract_pixel_size_ang(SOURCE_PATH) or 0.25
MATERIAL = 'WSSe'
D_SPACING_ANG = default_d_spacing_ang(get_material(MATERIAL))  # 1st-order plane spacing
print(f"pixel size = {PIXEL_SIZE_ANG:.4f} Å/px, d-spacing = {D_SPACING_ANG:.3f} Å ({MATERIAL})")

analysis = tiled.analyze(FRAME_IDX, ROW_IDX, COL_IDX, apply_hann=True, subtract_mean=SUBTRACT_MEAN)

expected_radius = TileFFTAnalysis.expected_ring_radius(
    tiled.tile_size, pixel_size_ang=PIXEL_SIZE_ANG, d_spacing_ang=D_SPACING_ANG
)
print(f"expected_radius = {expected_radius:.2f} px")

ring_radius = analysis.compute_ring_radius(expected_radius)
print(f"ring_radius = {ring_radius:.2f} px")
analysis.plot_ring_radius()

In [ ]:
angles, intensity = analysis.compute_angular_profile(dr=2.0, n_bins=360)
analysis.plot_angular_profile(n_sectors=6)

In [ ]:
twist_angle = analysis.compute_twist(n_folds=6)
analysis.plot_correlogram()
print(f"twist_angle = {twist_angle:.2f} degrees")

`analysis` now holds every cached stage result for this tile — useful for inspecting or reusing without recomputation:

In [ ]:
analysis